In [ ]:
import json
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from printer import Printer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_recall_curve, precision_score, recall_score, roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedKFold, cross_val_score


## Load data
Label meaning: 0 = negative (normal), 1 = positive (patient)

In [ ]:
data_path = '/data/jxiao/gas_data/gas_new/values_refine/RGB_diff_refine_label_new.csv'

diff = pd.read_csv(data_path, index_col='id')
diff_n2 = (np.sqrt(diff.iloc[:, :-1] ** 2)).T.groupby(np.arange(diff.shape[1] - 1) // 3).sum().T

X = diff.drop('label', axis=1, inplace=False).to_numpy()
X_n2 = diff_n2.to_numpy()
y = diff['label'].to_numpy().astype(int)
print(len(y), sum(y))

## Training configuration

In [4]:
USE_N2 = True
USE_SMOTE = False
N_SPLITS = 5
THRESHOLD = 0.5
RANDOM_SEED = 42
N_TRIALS = 50

best_params = None
best_score = None

feature_matrix = X_n2 if USE_N2 else X
feature_tag = 'n2' if USE_N2 else 'raw'
weight_dir = ''
if weight_dir:
    os.makedirs(weight_dir, exist_ok=True)
    logger = Printer(os.path.join(weight_dir, 'rf_log.txt'))

    with open(os.path.join(weight_dir, 'rf_config.txt'), 'w') as f:
        f.write(f'data_path: {data_path}\n')
        f.write(f'USE_N2: {USE_N2}\n')
        f.write(f'USE_SMOTE: {USE_SMOTE}\n')
        f.write(f'N_SPLITS: {N_SPLITS}\n')
        f.write(f'THRESHOLD: {THRESHOLD}\n')
        f.write(f'RANDOM_SEED: {RANDOM_SEED}\n')
        f.write(f'N_TRIALS: {N_TRIALS}\n')


def build_rf_model(params=None):
    if not params:
        return RandomForestClassifier(
            n_estimators=100,
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            max_features='sqrt',
            bootstrap=True,
            class_weight='balanced',
            random_state=RANDOM_SEED,
            n_jobs=-1,
        )
    return RandomForestClassifier(
        n_estimators=int(params['n_estimators']),
        max_depth=None if params['max_depth'] is None else int(params['max_depth']),
        min_samples_split=int(params['min_samples_split']),
        min_samples_leaf=int(params['min_samples_leaf']),
        max_features=params['max_features'],
        bootstrap=params['bootstrap'],
        class_weight='balanced',
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )


def build_pipeline(params):
    steps = []
    if USE_SMOTE:
        steps.append(('smote', SMOTE(random_state=RANDOM_SEED)))
    steps.append(('rf', build_rf_model(params)))
    return Pipeline(steps)


## Fit and predict with cross validation

In [ ]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED)

all_acc, all_f1, all_auc, all_precision, all_recall = [], [], [], [], []
all_fpr, all_tpr = [], []
all_precision_thre, all_recall_thre, all_thresholds = [], [], []
y_label_list, y_prob_list, y_pred_list = [], [], []

for fold, (train_idx, val_idx) in enumerate(skf.split(feature_matrix, y), start=1):
    print(f'===== Fold {fold} =====')

    X_train, X_val = feature_matrix[train_idx], feature_matrix[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    if USE_SMOTE:
        smote = SMOTE(random_state=RANDOM_SEED)
        X_train, y_train = smote.fit_resample(X_train, y_train)

    model = build_rf_model()
    model.fit(X_train, y_train)

    y_prob = model.predict_proba(X_val)[:, 1]
    y_pred = (y_prob > THRESHOLD).astype(int)

    acc = accuracy_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred, zero_division=0)
    auc = roc_auc_score(y_val, y_prob)
    precision = precision_score(y_val, y_pred, zero_division=0)
    recall = recall_score(y_val, y_pred, zero_division=0)
    fpr, tpr, _ = roc_curve(y_val, y_prob)
    precision_thre, recall_thre, thresholds = precision_recall_curve(y_val, y_prob)

    all_acc.append(acc)
    all_f1.append(f1)
    all_auc.append(auc)
    all_precision.append(precision)
    all_recall.append(recall)
    all_fpr.append(fpr)
    all_tpr.append(tpr)
    all_precision_thre.append(precision_thre)
    all_recall_thre.append(recall_thre)
    all_thresholds.append(thresholds)
    y_label_list.append(y_val.tolist())
    y_prob_list.append(y_prob.tolist())
    y_pred_list.append(y_pred.tolist())

    if weight_dir:
        with open(os.path.join(weight_dir, f'rf_split_fold_{fold}.json'), 'w') as f:
            json.dump({'train_idx': train_idx.tolist(), 'val_idx': val_idx.tolist()}, f)

        logger.write(
            f'===== Fold {fold} =====',
            f'Fold {fold} val results: ACC={acc:.3f}, F1={f1:.3f}, '
            f'precision={precision:.3f}, recall={recall:.3f}, AUC={auc:.3f}'
        )
        
    print(
        f'Fold {fold} val results: ACC={acc:.3f}, F1={f1:.3f}, '
        f'precision={precision:.3f}, recall={recall:.3f}, AUC={auc:.3f}'
    )


In [ ]:
## default params
print(f'===== Overall Results ({N_SPLITS}-Fold Avg) =====')
print(f'ACC={np.mean(all_acc):.3f} ± {np.std(all_acc):.3f}')
print(f'F1 ={np.mean(all_f1):.3f} ± {np.std(all_f1):.3f}')
print(f'AUC={np.mean(all_auc):.3f} ± {np.std(all_auc):.3f}')
print(f'Precision={np.mean(all_precision):.3f} ± {np.std(all_precision):.3f}')
print(f'Recall={np.mean(all_recall):.3f} ± {np.std(all_recall):.3f}')

if weight_dir:
    logger.write('===== Overall Results =====')
    logger.write(f'ACC={np.mean(all_acc):.4f} ± {np.std(all_acc):.4f}')
    logger.write(f'F1 ={np.mean(all_f1):.4f} ± {np.std(all_f1):.4f}')
    logger.write(f'AUC={np.mean(all_auc):.4f} ± {np.std(all_auc):.4f}')
    logger.write(f'Precision={np.mean(all_precision):.4f} ± {np.std(all_precision):.4f}')
    logger.write(f'Recall={np.mean(all_recall):.4f} ± {np.std(all_recall):.4f}')

## Save results

In [17]:
js_val = {}
js_val['threshold'] = THRESHOLD
js_val['best_params'] = best_params
js_val['best_score'] = best_score
js_val['accuracy'] = all_acc
js_val['f1_score'] = all_f1
js_val['auc'] = all_auc
js_val['precision'] = all_precision
js_val['recall'] = all_recall
js_val['fpr'] = [fpr.tolist() for fpr in all_fpr]
js_val['tpr'] = [tpr.tolist() for tpr in all_tpr]
js_val['precision_thre'] = [prec.tolist() for prec in all_precision_thre]
js_val['recall_thre'] = [rec.tolist() for rec in all_recall_thre]
js_val['thresholds'] = [thre.tolist() for thre in all_thresholds]
js_val['y_label'] = y_label_list
js_val['y_prob'] = y_prob_list
js_val['y_pred'] = y_pred_list

with open(os.path.join(weight_dir, 'RF_val.json'), 'w') as f:
    json.dump(js_val, f)


In [ ]:
fig, ax = plt.subplots(1, N_SPLITS, figsize=(20, 4), sharey=True)
if N_SPLITS == 1:
    ax = [ax]

for i in range(N_SPLITS):
    ax[i].plot(all_fpr[i], all_tpr[i], color='darkorange', lw=2, label=f'ROC curve (AUC = {all_auc[i]:.3f})')
    ax[i].plot([0, 1], [0, 1], color='navy', lw=1.5, linestyle='--')
    ax[i].set_title(f'Fold {i + 1}', fontsize=14)
    ax[i].set_xlabel('False Positive Rate', fontsize=14)
    ax[i].legend(loc='lower right')

fig.suptitle('Random Forest ROC Curve', fontsize=14, fontweight='bold')
fig.supylabel('True positive rate', fontsize=14)
plt.tight_layout(rect=[0.005, -0.05, 1, 0.999])
plt.show()
